<a href="https://colab.research.google.com/github/Jazperism/Applied-ML-S26-Final/blob/main/Applied-ML-S26-Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# download the customer churn.csv
!wget -O Churn.csv https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv

#loading in data
import pandas as pd

tcchurn = pd.read_csv('Churn.csv')
tcchurn.head()

print(tcchurn.head())

--2026-04-19 02:40:13--  https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 970457 (948K) [text/plain]
Saving to: ‘Churn.csv’

Churn.csv           100%[===================>] 947.71K  --.-KB/s    in 0.1s    

2026-04-19 02:40:13 (8.26 MB/s) - ‘Churn.csv’ saved [970457/970457]

   customerID  gender  SeniorCitizen  ... MonthlyCharges TotalCharges  Churn
0  7590-VHVEG  Female              0  ...          29.85        29.85     No
1  5575-GNVDE    Male              0  ...          56.95       1889.5     No
2  3668-QPYBK    Male              0  ...          53.85       108.15    Yes
3  7795-CFOCW    Male              0  ...          42.30      1840.75     No
4  9

In [ ]:
# @title Encoding

from sklearn.preprocessing import LabelEncoder, OneHotEncoder

# label encode the target variable 'churn'
le = LabelEncoder()

# added this in order to check if 'Churn' column exists and is of object type (string 'Yes'/'No') before encoding
# this prevents errors if 'Churn' is already encoded or missing, which its probably already encoded if ur reading this
if 'Churn' in tcchurn.columns and tcchurn['Churn'].dtype == 'object':
    tcchurn['Churn'] = le.fit_transform(tcchurn['Churn'])
else:
    print("warning: 'Churn' column not found or already encoded for label encoding")

# list of columns to one-hot encode, including 'gender' and other remaining categorical features
cols_to_encode = [
    'Contract', 'PaymentMethod', 'InternetService', 'gender', 'Partner', 'Dependents',
    'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling'
]

# filter for columns that actually exist in the DataFrame
existing_cols_to_encode = [col for col in cols_to_encode if col in tcchurn.columns]

if existing_cols_to_encode:
    tcchurn = pd.get_dummies(tcchurn, columns=existing_cols_to_encode)
else:
    print("warning: Columns for one-hot encoding already processed or not found.")

In [ ]:
# @title Standardizing

from sklearn.preprocessing import StandardScaler
import numpy as np

scaler = StandardScaler()

# scaling the continous numerical feature
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

# Handle empty strings in 'TotalCharges' first, converting them to NaN
# and then filling NaN with 0, as an empty 'TotalCharges' likely means no charges.
tcchurn['TotalCharges'] = tcchurn['TotalCharges'].replace(' ', np.nan).astype(float)
tcchurn['TotalCharges'] = tcchurn['TotalCharges'].fillna(0)

# Now apply scaling to the continuous numerical features
tcchurn[num_cols] = scaler.fit_transform(tcchurn[num_cols])

In [ ]:
# testing logisitc regression model for classification report/comparisions to kNN
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# defining our x (feature) and y (target)
X = tcchurn.drop(['customerID', 'Churn'], axis=1)
y = tcchurn['Churn']

# train/test splits
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# inialize and train the first model
log_model = LogisticRegression()
log_model.fit(X_train, y_train)

# perdict and evaluate
prediction = log_model.predict(X_test)
print(classification_report(y_test, prediction))

              precision    recall  f1-score   support

           0       0.86      0.90      0.88      1036
           1       0.69      0.60      0.64       373

    accuracy                           0.82      1409
   macro avg       0.77      0.75      0.76      1409
weighted avg       0.82      0.82      0.82      1409



In [ ]:
# testing kNN model for
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report

# initate model
knn_model = KNeighborsClassifier(n_neighbors=5)

# train the model
knn_model.fit(X_train, y_train)

# predict and evaluate
knn_prediction = knn_model.predict(X_test)

# evaluate
print(classification_report(y_test, knn_prediction))

              precision    recall  f1-score   support

           0       0.84      0.86      0.85      1036
           1       0.59      0.56      0.57       373

    accuracy                           0.78      1409
   macro avg       0.72      0.71      0.71      1409
weighted avg       0.78      0.78      0.78      1409



In [ ]:
from sklearn.model_selection import GridSearchCV

# now testing different k values and different distance mertics
knn_param_grid = {
    'n_neighbors': [3, 5, 7, 19],
    'metric': ['euclidean', 'manhattan']
}

knn_grid = GridSearchCV(KNeighborsClassifier(), knn_param_grid, cv=5, scoring='f1')
knn_grid.fit(X_train, y_train)

# print
print(f"Best k-NN Params: {knn_grid.best_params_}")
print(f"Best k-NN F1 Score: {knn_grid.best_score_:.2f}")

log_param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l2'],
    'solver': ['liblinear', 'lbfgs']
}

log_grid = GridSearchCV(LogisticRegression(max_iter=1000), log_param_grid, cv=5, scoring='f1')
log_grid.fit(X_train, y_train)

print(f"Best Logistic Regression Params: {log_grid.best_params_}")
print(f"Best Logistic Regression F1 Score: {log_grid.best_score_:.2f}")

Best k-NN Params: {'metric': 'euclidean', 'n_neighbors': 19}
Best k-NN F1 Score: 0.59
Best Logistic Regression Params: {'C': 10, 'penalty': 'l2', 'solver': 'liblinear'}
Best Logistic Regression F1 Score: 0.59


*Put analysis for different churning models here*

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# download the credit card fraud dataset
!wget -O creditcard.csv https://raw.githubusercontent.com/nsethi31/Kaggle-Data-Credit-Card-Fraud-Detection/master/creditcard.csv

# load the data
df_fraud = pd.read_csv('creditcard.csv')

print(f"dataset shape:{df_fraud.shape}")
print(df_fraud.isnull().sum().max()) # this verifies the 'no nulls'






--2026-04-19 04:08:01--  https://raw.githubusercontent.com/nsethi31/Kaggle-Data-Credit-Card-Fraud-Detection/master/creditcard.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 102634230 (98M) [text/plain]
Saving to: ‘creditcard.csv’

creditcard.csv      100%[===================>]  97.88M   120MB/s    in 0.8s    

2026-04-19 04:08:04 (120 MB/s) - ‘creditcard.csv’ saved [102634230/102634230]

dataset shape:(284807, 31)
0


In [ ]:
# check the imbalance ratio ( just to confirm the high imbalance )
imbalance = df_fraud['Class'].value_counts(normalize=True) * 100
print(f"Percentage of Normal (0): {imbalance[0]:.2f}%")
print(f"Percentage of Fraud (1): {imbalance[1]:.2f}%")

Percentage of Normal (0): 99.83%
Percentage of Fraud (1): 0.17%


In [ ]:
# establishing baseline standarization before adjusting hyper parameters
from sklearn.preprocessing import StandardScaler

# initialize the scaler
scaler = StandardScaler()

# scale Time and Amount
df_fraud['scaled_amount'] = scaler.fit_transform(df_fraud['Amount'].values.reshape(-1,1))
df_fraud['scaled_time'] = scaler.fit_transform(df_fraud['Time'].values.reshape(-1,1))

# drop original columns and define X and y
X_fraud = df_fraud.drop(['Class', 'Amount', 'Time'], axis=1)
y_fraud = df_fraud['Class']

# stratify=y ensures both sets have the 0.17% fraud represented
X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_fraud, y_fraud, test_size=0.2, random_state=42, stratify=y_fraud
)

In [ ]:
# baseline model for LogisticRegression
from sklearn.linear_model import LogisticRegression

# initialize and fit
log_fraud_baseline = LogisticRegression(class_weight='balanced', max_iter=1000)
log_fraud_baseline.fit(X_train_f, np.ravel(y_train_f))

# evaluate
print("Logistic Regression (Baseline) Report:")
print(classification_report(y_test_f, log_fraud_baseline.predict(X_test_f)))

Logistic Regression (Baseline) Report:
              precision    recall  f1-score   support

           0       1.00      0.98      0.99     56864
           1       0.06      0.92      0.11        98

    accuracy                           0.98     56962
   macro avg       0.53      0.95      0.55     56962
weighted avg       1.00      0.98      0.99     56962



In [ ]:
# baseline model for LD
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# initialize and fit
lda_fraud_baseline = LinearDiscriminantAnalysis()
lda_fraud_baseline.fit(X_train_f, np.ravel(y_train_f))

# evaluate
print("LDA (Baseline) Report:")
print(classification_report(y_test_f, lda_fraud_baseline.predict(X_test_f)))

LDA (Baseline) Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.82      0.81      0.81        98

    accuracy                           1.00     56962
   macro avg       0.91      0.90      0.91     56962
weighted avg       1.00      1.00      1.00     56962



In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# adjusting/defining our new parameters
lda_params = {
    'solver': ['lsqr', 'eigen'],
    'shrinkage': [None, 'auto', 0.1, 0.5, 0.9]
}

# now we run the GridSearch, using scoring='f1' because we need to balance precision and recall
lda_grid = GridSearchCV(LinearDiscriminantAnalysis(), lda_params, cv=5, scoring='f1')
lda_grid.fit(X_train_f, np.ravel(y_train_f))

print(f"Best LDA Params: {lda_grid.best_params_}")
print(f"Best LDA F1 Score: {lda_grid.best_score_:.2f}")


Best LDA Params: {'shrinkage': 0.1, 'solver': 'lsqr'}
Best LDA F1 Score: 0.81


In [ ]:
# defining/adjusting new parameters
log_params = {
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['liblinear', 'lbfgs']
}

# run gridsearch, IMPORTANT to keep class_weight='balanced' here so it doesn't ignore fraud all together
log_grid = GridSearchCV(LogisticRegression(class_weight='balanced', max_iter=1000), log_params, cv=5, scoring='f1')
log_grid.fit(X_train_f, np.ravel(y_train_f))

print(f"Best Logistic Regression Params: {log_grid.best_params_}")
print(f"Best Logistic Regression F1 Score: {log_grid.best_score_:.2f}")

Best Logistic Regression Params: {'C': 0.01, 'solver': 'lbfgs'}
Best Logistic Regression F1 Score: 0.12
